# MSEA-Net: Interactive Clinical Diagnostic & Uncertainty Demo
**Author**: Abdullah Rubab (`rubab2712@gmail.com`)  
**Paper**: *MSEA-Net: Multi-Scale Evidential Attention Network with Uncertainty Calibration for Gastrointestinal Disease Classification and Polyp Segmentation*

This notebook demonstrates how to load a trained MSEA-Net checkpoint, execute inference on an endoscopy image, quantify Dirichlet epistemic uncertainty, and generate Grad-CAM++ saliency overlays.

In [ ]:
# 1. Imports and environment setup
import torch
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import torchvision.transforms as T

from msea_net.models.msea_net import MSEANet
from msea_net.data.dataset import DEFAULT_CLASSES, IMAGENET_MEAN, IMAGENET_STD
from msea_net.xai.explainers import generate_gradcam_heatmap

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# 2. Instantiate and load trained model weights
model = MSEANet(num_classes=8, pretrained=False).to(device)
# checkpoint = torch.load('msea_net_results/msea_net_seed777_best.pth', map_location=device)
# model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print("MSEA-Net initialized successfully!")

In [ ]:
# 3. Test Dummy Forward Pass & Uncertainty Output
dummy_image = torch.randn(1, 3, 224, 224).to(device)
with torch.no_grad():
    out = model(dummy_image)

probs = out['probs'].cpu().numpy()[0]
unc = out['uncertainty'].cpu().numpy()[0]
pred_class = DEFAULT_CLASSES[np.argmax(probs)]

print(f"Predicted Class:      {pred_class}")
print(f"Confidence:           {np.max(probs)*100:.2f}%")
print(f"Epistemic Uncertainty: {unc:.4f} (0=Certain, 8=Total Ambiguity)")
print(f"Dirichlet Alpha Vector: {out['alpha'].cpu().numpy()[0]}")